# Smart PDF to Markdown Converter - Extract FIGURES Only

This notebook intelligently extracts **only FIGURE images** (like FIGURE 1-1, FIGURE 2-1, etc.) along with their captions, while **skipping decorative images**.

**Key Features:**
- ✅ Extracts only images with "FIGURE" captions
- ✅ Captures figure numbers and descriptions
- ✅ Filters out small decorative images
- ✅ Saves figures with meaningful names (e.g., `figure_1-1.png`)
- ✅ Includes complete caption text in markdown
- ✅ GPU-accelerated for OCR tasks

**Perfect for textbooks and academic PDFs!**

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install -q pymupdf PyMuPDF Pillow

print("✅ All packages installed successfully!")

## Step 2: Import Libraries

In [ ]:
import fitz  # PyMuPDF
import os
import io
from pathlib import Path
from PIL import Image
import json
from datetime import datetime
import re

print("✅ Libraries imported successfully!")

## Step 3: Mount Google Drive (Optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted at /content/drive")

## Step 4: Upload PDF File

In [ ]:
# Method A: Upload from computer
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# Method B: Use from Google Drive (uncomment if using)
# pdf_path = '/content/drive/MyDrive/your-pdf-file.pdf'

print(f"✅ PDF file ready: {pdf_path}")

## Step 5: Smart Figure Extractor Class

In [ ]:
class SmartFigureExtractor:
    """
    Intelligently extracts only FIGURE images with captions from PDFs.
    Filters out decorative images and small graphics.
    """
    
    def __init__(self, pdf_path, output_dir="output", min_image_size=10000):
        """
        Initialize the smart figure extractor.
        
        Args:
            pdf_path (str): Path to the PDF file
            output_dir (str): Directory to save output files
            min_image_size (int): Minimum image area in pixels² (default: 10000)
        """
        self.pdf_path = pdf_path
        self.output_dir = output_dir
        self.images_dir = os.path.join(output_dir, "images")
        self.min_image_size = min_image_size
        
        # Create output directories
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.images_dir, exist_ok=True)
        
        # Open PDF
        self.doc = fitz.open(pdf_path)
        self.figure_counter = 0
        self.skipped_counter = 0
        
    def clean_text(self, text):
        """Clean and normalize extracted text."""
        text = re.sub(r'\s+', ' ', text)
        return text.strip()
    
    def find_figure_caption(self, page, img_bbox, search_distance=150):
        """
        Find the caption text near an image, looking for FIGURE patterns.
        
        Args:
            page: PyMuPDF page object
            img_bbox: Image bounding box (x0, y0, x1, y1)
            search_distance: How far to search for caption (in points)
            
        Returns:
            tuple: (figure_number, caption_text) or (None, None)
        """
        blocks = page.get_text("dict")["blocks"]
        
        img_x0, img_y0, img_x1, img_y1 = img_bbox
        caption_candidates = []
        
        for block in blocks:
            if block["type"] == 0:  # Text block
                block_bbox = block["bbox"]
                block_x0, block_y0, block_x1, block_y1 = block_bbox
                
                # Calculate distance from image
                vertical_distance = min(
                    abs(block_y0 - img_y1),  # Distance below image
                    abs(img_y0 - block_y1)   # Distance above image
                )
                
                # Check horizontal overlap
                horizontal_overlap = min(img_x1, block_x1) - max(img_x0, block_x0)
                
                if vertical_distance < search_distance and horizontal_overlap > 0:
                    # Extract text from block
                    text = ""
                    for line in block.get("lines", []):
                        for span in line.get("spans", []):
                            text += span.get("text", "") + " "
                    
                    text = self.clean_text(text)
                    
                    # Look for FIGURE pattern (case-insensitive)
                    # Matches: FIGURE 1-1, Figure 2.3, FIGURE 1-8, etc.
                    figure_patterns = [
                        r'FIGURE\s+([\d]+[-\.][\d]+[A-Z]?)',  # FIGURE 1-1, FIGURE 2.3
                        r'FIGURE\s+([\d]+)',                   # FIGURE 1
                        r'Fig\.?\s+([\d]+[-\.][\d]+[A-Z]?)',  # Fig. 1-1
                        r'Fig\.?\s+([\d]+)'                    # Fig 1
                    ]
                    
                    for pattern in figure_patterns:
                        match = re.search(pattern, text, re.IGNORECASE)
                        if match:
                            caption_candidates.append({
                                'text': text,
                                'figure_num': match.group(1),
                                'distance': vertical_distance,
                                'y_pos': block_y0
                            })
                            break
        
        # Return the closest caption
        if caption_candidates:
            closest = min(caption_candidates, key=lambda x: x['distance'])
            return closest['figure_num'], closest['text']
        
        return None, None
    
    def is_figure_image(self, img_bbox, page_width, page_height):
        """
        Determine if an image is likely a figure (not decorative).
        
        Args:
            img_bbox: Image bounding box
            page_width: Page width
            page_height: Page height
            
        Returns:
            bool: True if image is likely a figure
        """
        x0, y0, x1, y1 = img_bbox
        width = x1 - x0
        height = y1 - y0
        area = width * height
        
        # Filter out very small images (likely decorative)
        if area < self.min_image_size:
            return False
        
        # Filter out very thin images (likely borders/lines)
        if width > 0 and height > 0:
            aspect_ratio = max(width, height) / min(width, height)
            if aspect_ratio > 20:
                return False
        
        return True
    
    def extract_figures_from_page(self, page, page_num):
        """
        Extract only figure images from a PDF page with their captions.
        
        Args:
            page: PyMuPDF page object
            page_num (int): Page number
            
        Returns:
            list: List of dictionaries with figure info
        """
        image_list = page.get_images(full=True)
        extracted_figures = []
        
        page_width = page.rect.width
        page_height = page.rect.height
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            
            try:
                # Get image position on page
                img_rects = page.get_image_rects(xref)
                if not img_rects:
                    continue
                
                img_bbox = img_rects[0]  # Use first occurrence
                
                # Check if this is a figure (not decorative)
                if not self.is_figure_image(img_bbox, page_width, page_height):
                    self.skipped_counter += 1
                    print(f"  ⏭️  Skipping small/decorative image on page {page_num}")
                    continue
                
                # Look for figure caption
                figure_num, caption = self.find_figure_caption(page, img_bbox)
                
                if not figure_num:
                    self.skipped_counter += 1
                    print(f"  ⏭️  Skipping image without FIGURE caption on page {page_num}")
                    continue
                
                # Extract image
                base_image = self.doc.extract_image(xref)
                image_bytes = base_image["image"]
                image_ext = base_image["ext"]
                
                # Generate filename based on figure number
                self.figure_counter += 1
                safe_fig_num = figure_num.replace('-', '_').replace('.', '_').replace(' ', '_')
                image_filename = f"figure_{safe_fig_num}.{image_ext}"
                image_path = os.path.join(self.images_dir, image_filename)
                
                # Save image
                with open(image_path, "wb") as img_file:
                    img_file.write(image_bytes)
                
                extracted_figures.append({
                    'filename': image_filename,
                    'figure_num': figure_num,
                    'caption': caption,
                    'page': page_num
                })
                
                print(f"  ✅ Extracted FIGURE {figure_num}: {image_filename}")
                # Truncate caption for display
                caption_preview = caption[:100] + "..." if len(caption) > 100 else caption
                print(f"     Caption: {caption_preview}")
                
            except Exception as e:
                print(f"  ⚠️ Error processing image {img_index} on page {page_num}: {e}")
        
        return extracted_figures
    
    def extract_text_from_page(self, page):
        """
        Extract text from a page with structure detection.
        """
        blocks = page.get_text("dict")["blocks"]
        markdown_text = ""
        
        for block in blocks:
            if block["type"] == 0:  # Text block
                text = ""
                font_size = 0
                
                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        text += span.get("text", "")
                        if span.get("size", 0) > font_size:
                            font_size = span.get("size", 0)
                    text += " "
                
                text = self.clean_text(text)
                
                if text:
                    # Detect if it's a heading based on font size
                    if font_size > 18:
                        markdown_text += f"\n# {text}\n\n"
                    elif font_size > 14:
                        markdown_text += f"\n## {text}\n\n"
                    elif font_size > 12:
                        markdown_text += f"\n### {text}\n\n"
                    else:
                        markdown_text += f"{text}\n\n"
        
        return markdown_text
    
    def convert_to_markdown(self):
        """
        Convert entire PDF to Markdown with figure images only.
        
        Returns:
            str: Path to the generated markdown file
        """
        print(f"\n🚀 Starting smart extraction from: {self.pdf_path}")
        print(f"📄 Total pages: {len(self.doc)}")
        print(f"⚙️  Min image size: {self.min_image_size} pixels²\n")
        
        markdown_content = f"# {Path(self.pdf_path).stem}\n\n"
        markdown_content += f"*Extracted on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*\n\n"
        markdown_content += "*Note: Only FIGURE images with captions are included*\n\n"
        markdown_content += "---\n\n"
        
        all_figures = []
        
        # Process each page
        for page_num in range(len(self.doc)):
            print(f"\n📖 Processing page {page_num + 1}/{len(self.doc)}...")
            
            page = self.doc[page_num]
            
            # Add page marker
            markdown_content += f"\n## Page {page_num + 1}\n\n"
            
            # Extract text
            text = self.extract_text_from_page(page)
            markdown_content += text
            
            # Extract figure images with captions
            figures = self.extract_figures_from_page(page, page_num + 1)
            all_figures.extend(figures)
            
            # Add figure references to markdown
            if figures:
                markdown_content += "\n### 📊 Figures on This Page\n\n"
                for fig in figures:
                    markdown_content += f"#### FIGURE {fig['figure_num']}\n\n"
                    markdown_content += f"![Figure {fig['figure_num']}](images/{fig['filename']})\n\n"
                    markdown_content += f"**Caption:** *{fig['caption']}*\n\n"
                    markdown_content += "---\n\n"
        
        # Add figure index at the end
        if all_figures:
            markdown_content += "\n\n# 📑 Figure Index\n\n"
            markdown_content += "| Figure | Caption | Page |\n"
            markdown_content += "|--------|---------|------|\n"
            for fig in all_figures:
                caption_short = fig['caption'][:60] + "..." if len(fig['caption']) > 60 else fig['caption']
                markdown_content += f"| FIGURE {fig['figure_num']} | {caption_short} | {fig['page']} |\n"
        
        # Save markdown file
        output_filename = Path(self.pdf_path).stem + "_figures.md"
        output_path = os.path.join(self.output_dir, output_filename)
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(markdown_content)
        
        print(f"\n" + "="*60)
        print("✅ EXTRACTION COMPLETE!")
        print("="*60)
        print(f"📝 Markdown file: {output_path}")
        print(f"✅ Figures extracted: {self.figure_counter}")
        print(f"⏭️  Images skipped: {self.skipped_counter}")
        print(f"📁 Images directory: {self.images_dir}")
        
        return output_path
    
    def generate_summary(self):
        """
        Generate a summary JSON file with extraction statistics.
        """
        summary = {
            "pdf_file": self.pdf_path,
            "total_pages": len(self.doc),
            "figures_extracted": self.figure_counter,
            "images_skipped": self.skipped_counter,
            "min_image_size": self.min_image_size,
            "output_directory": self.output_dir,
            "extraction_date": datetime.now().isoformat(),
        }
        
        summary_path = os.path.join(self.output_dir, "extraction_summary.json")
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        
        return summary
    
    def close(self):
        """Close the PDF document."""
        self.doc.close()

print("✅ SmartFigureExtractor class defined!")

## Step 6: Configure Extraction Settings

Adjust these settings to control figure extraction:

In [ ]:
# ===== EXTRACTION SETTINGS =====

# Output directory
output_directory = "smart_extraction_output"

# Minimum image area (in pixels²) to be considered a figure
# Adjust based on your PDF:
#   - 5000 = Extract smaller figures
#   - 10000 = Default, good for most textbooks
#   - 20000 = Only large figures
MIN_IMAGE_SIZE = 10000

print("✅ Extraction settings configured!")
print(f"   📁 Output: {output_directory}")
print(f"   📏 Min size: {MIN_IMAGE_SIZE} pixels²")
print(f"   🎯 Target: Images with 'FIGURE' captions only")

## Step 7: Run the Smart Extraction

In [ ]:
# Create extractor instance
extractor = SmartFigureExtractor(
    pdf_path, 
    output_dir=output_directory,
    min_image_size=MIN_IMAGE_SIZE
)

# Convert PDF to Markdown (extracts only FIGURE images with captions)
markdown_file = extractor.convert_to_markdown()

# Generate summary
summary = extractor.generate_summary()

# Close the PDF
extractor.close()

print("\n" + "="*60)
print("📊 DETAILED SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"  {key}: {value}")

## Step 8: Preview Extracted Figures

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

# Get all extracted figures
figure_files = sorted(glob.glob(os.path.join(output_directory, "images", "figure_*")))

if figure_files:
    print(f"📷 Found {len(figure_files)} extracted figures\n")
    
    # List all figures
    for i, fig_path in enumerate(figure_files, 1):
        print(f"  {i}. {os.path.basename(fig_path)}")
    
    # Display first 6 figures
    num_to_display = min(6, len(figure_files))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(figure_files[:num_to_display]):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(os.path.basename(img_path), fontsize=10, fontweight='bold')
        axes[idx].axis('off')
    
    # Hide unused subplots
    for idx in range(num_to_display, 6):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Displayed {num_to_display} of {len(figure_files)} figures")
else:
    print("⚠️ No figures were extracted. Try adjusting MIN_IMAGE_SIZE.")

## Step 9: Preview the Markdown File

In [ ]:
# Read and display portion of the markdown file
with open(markdown_file, "r", encoding="utf-8") as f:
    content = f.read()

print("📝 MARKDOWN PREVIEW:")
print("="*70)
print(content[:3000])  # Show first 3000 characters
print("\n...\n")
print(f"(Full document: {len(content)} characters, {content.count('FIGURE')} figure references)")

## Step 10: Download Results

Download the markdown file and extracted figures as a zip file.

In [ ]:
import shutil

# Create a zip file with all results
zip_filename = "extracted_figures"
shutil.make_archive(zip_filename, 'zip', output_directory)

print(f"📦 Created: {zip_filename}.zip")
print(f"   Contains: Markdown file + {summary['figures_extracted']} figure images")

# Download the zip file
from google.colab import files
files.download(f"{zip_filename}.zip")

print("\n✅ Download started! Check your browser's download folder.")

## 🎯 Tips for Better Results

If you're not getting the figures you expect:

1. **Adjust MIN_IMAGE_SIZE**: Try lower values (5000) for smaller figures
2. **Check caption format**: The extractor looks for "FIGURE", "Fig.", or "Fig" followed by a number
3. **Verify PDF quality**: Scanned PDFs may need OCR preprocessing
4. **Check the summary**: Review `figures_extracted` vs `images_skipped` counts

### Common Issues:

- **No figures extracted**: MIN_IMAGE_SIZE might be too high, or captions aren't detected
- **Too many decorative images**: Increase MIN_IMAGE_SIZE
- **Missing some figures**: Captions might use different format (check the PDF manually)

## Optional: Re-run with Different Settings

If you want to try different extraction settings, just change the values in Step 6 and re-run from there!